# GenAI + RAG: explaining the maps in plain language

Member 4, step 2 (slide 22). `urbanheat.app.assistant`:
- **Facts** come from the model and the XAI for the place or spot the user selected (measured and modelled
  difference from the city's typical surface temperature, the reasons in °C). The language model never makes up numbers.
- **Retrieval (RAG):** `docs/KNOWLEDGE.md`, written for dashboard users (what LST, NDVI, NDBI… mean, the project's
  results and limits), is split into one passage per section and indexed with TF-IDF (small, domain-specific corpus,
  no embedding service needed). The team documents are left out: their developer jargon misled the answers. The best
  4 passages go into the prompt and are cited as [1]–[4].
- **Generation:** Claude (`claude-opus-5-5`, low effort, server-side fallback if a request is declined) writes a
  grounded answer of under 200 words, with the caveats (surface, not air temperature; model estimates, not causes).
- **Offline:** without Claude credentials the assistant still answers the common questions from the same facts: it
  recognises the kind of question (is it hot or cool, why, would trees help, how sure, what does a word mean) and
  answers it in plain words. To enable Claude, set `ANTHROPIC_API_KEY` (or run `ant auth login`).

In [1]:
import warnings, base64, io, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from fastapi.testclient import TestClient
from urbanheat.app import assistant
from urbanheat.app.server import create_app, PRESETS
warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 90
app = create_app()                     # dataset from the config, model runs/m3_lite16_kd
ws, api = app.state.ws, TestClient(app)
PID = "EMIT_243"
summary = api.get(f"/api/summary/{PID}").json()
HOT = summary["spots"]["hottest"]      # the dashboard's "Hottest spot" button
print(ws.run, "|", len(ws.info), "images |", summary["name"], summary["region"], summary["date"], "| hottest spot", HOT)

m3_lite16_kd | 422 images | Barranquilla Atlantico 2025-03-11 | hottest spot {'lat': 10.961177759124956, 'lng': -74.84110646212541, 'diff_c': 3.7863082885742188}


## The knowledge base

In [2]:
kb = ws.knowledge
pd.DataFrame([(s, t, len(x)) for s, t, x in kb.passages], columns=["source", "passage", "chars"]).groupby("source").agg(
    passages=("passage", "count"), chars=("chars", "sum"))

,passages,chars
source,,
KNOWLEDGE.md,18,7236


## Retrieval for the questions on slide 22

In [3]:
for q in ["Why is this area predicted to be hot?", "Which factors influence this prediction?",
          "What does a high NDBI value indicate?", "Is this the air temperature people feel?",
          "Would planting trees here cool it, and can I trust that?"]:
    print(q); [print(f"   {h['score']:.2f}  {h['title']}  ({h['source']})") for h in kb.search(q, 3)]

Why is this area predicted to be hot?
   0.11  How accurate is the model?  (KNOWLEDGE.md)
   0.08  What is "heat seen on other days"?  (KNOWLEDGE.md)
   0.06  How can a hot place be cooled?  (KNOWLEDGE.md)
Which factors influence this prediction?
   0.15  What makes places hot in this data?  (KNOWLEDGE.md)
   0.08  What do the "what if" buttons do?  (KNOWLEDGE.md)
   0.05  How does the dashboard explain a spot?  (KNOWLEDGE.md)
What does a high NDBI value indicate?
   0.27  What does NDBI (built-up surfaces) mean?  (KNOWLEDGE.md)
   0.07  How does the model work?  (KNOWLEDGE.md)
   0.05  What do the "what if" buttons do?  (KNOWLEDGE.md)
Is this the air temperature people feel?
   0.28  Is this the air temperature? (land surface temperature, LST)  (KNOWLEDGE.md)
   0.13  How can a hot place be cooled?  (KNOWLEDGE.md)
   0.10  Can I trust the "what if" results?  (KNOWLEDGE.md)
Would planting trees here cool it, and can I trust that?
   0.13  Can I trust the "what if" results?  (KNOWLEDGE.

## Facts for the hottest spot, and the prompt Claude receives

In [4]:
facts = ws.explain(PID, HOT["lat"], HOT["lng"], 600)["facts"]
hits = kb.search(f"Why is this spot hot? {facts['top_factors']}", 4)
print(assistant.SYSTEM, "\n")
print("FACTS\n" + assistant.format_facts(facts))
print("\nPASSAGES\n" + "\n".join(f"[{i}] {h['title']}: {h['text'][:160]}..." for i, h in enumerate(hits, 1)))

You explain urban heat maps to city planners and students. Answer from the FACTS (model outputs for the place the user selected) and the numbered PASSAGES only. Cite passages as [1], [2]. Give numbers with units (deg C). Say plainly that predictions are land surface temperature, not air temperature, and that attributions and scenarios are model estimates, not proof of cause. If the facts and passages do not answer the question, say so. Write for people with no science background: answer the question in the first sentence, use short sentences and everyday words, and explain any technical term in a few words. Keep it under 150 words. 

FACTS
- place: Barranquilla, Atlantico
- date: 2025-03-11
- spot_radius_m: 600
- pixels: 321
- city_typical_surface_c: 45.2
- measured_surface_c: 46.5
- measured_vs_typical_c: 1.27
- model_vs_typical_c: 0.52
- average_place_c: -0.13
- top_factors: Heat seen on other days +0.55 °C, Surface colour and brightness -0.09 °C, Built-up or bare surfaces +0.07 °C



## Answers

In [5]:
for q in ["Is this place cool?", "Why is this spot hot?", "Would trees help here?", "How sure is the model?",
          "What does a high NDBI value indicate?"]:
    r = api.post("/api/ask", json={"question": q, "pid": PID, "lat": HOT["lat"], "lng": HOT["lng"]}).json()
    print(f"Q: {q}\n[{r['mode']}] {r['answer']}\nsources: {[s['title'] for s in r['sources']]}\n")

Q: Is this place cool?
[offline] No. The ground in this circle is 46.5 °C, 1.3 °C hotter than the typical 45.2 °C of Barranquilla, Atlantico. The biggest reasons, according to the model: Heat seen on other days +0.55 °C, Surface colour and brightness -0.09 °C, Built-up or bare surfaces +0.07 °C. (This is the temperature of the ground seen from space, not the air; the model gives estimates, not proof.)
sources: []

Q: Why is this spot hot?
[offline] The model estimates this circle at +0.5 °C compared with the rest of Barranquilla, Atlantico. Starting from an average place (-0.1 °C), the biggest reasons are: Heat seen on other days +0.55 °C, Surface colour and brightness -0.09 °C, Built-up or bare surfaces +0.07 °C. 'Heat seen on other days' means this spot was also hot (or cool) in other satellite pictures. (This is the temperature of the ground seen from space, not the air; the model gives estimates, not proof.)
sources: []

Q: Would trees help here?
[offline] Use step 4, 'What if this

This notebook ran without Claude credentials, so the answers above are the **offline** mode. With
`ANTHROPIC_API_KEY` set, the same call returns Claude's wording of the same facts and passages
(`mode: "claude"`). The API path is covered by `tests/test_app.py::test_ask_uses_claude_when_available`.